In [4]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. THE STATE (Our Multi-Agent Memory)
# ==========================================
class AgentState(TypedDict):
    topic: str
    current_draft: str
    feedback: str
    quality_score: int
    revision_count: int

# ==========================================
# 2. THE NODES (The Workers)
# ==========================================

def writer_agent(state: AgentState):
    print(f"\n✍️ [Writer] Current Revision: {state['revision_count'] + 1}")
    
    # The writer adapts its output based on whether it received feedback yet
    if not state["feedback"]:
        draft = f"Our new software tool makes databases run super fast."
        print(f"   Draft 1: '{draft}'")
    else:
        print(f"   Received Feedback: \"{state['feedback']}\"")
        draft = "⚡ QuantumDB: Rocket-fuel your SQL queries with zero latency."
        print(f"   Draft 2 (Revised): '{draft}'")
        
    return {
        "current_draft": draft,
        "revision_count": state["revision_count"] + 1
    }


def critic_agent(state: AgentState):
    print("🧐 [Critic] Analyzing the latest draft...")
    draft = state["current_draft"]
    
    # The Critic enforces strict rules
    if "⚡" not in draft or "latency" not in draft:
        score = 5
        feedback = "Too boring. Needs an emoji, a cooler product name, and the word 'latency'."
    else:
        score = 9
        feedback = "Perfect. This hits all our enterprise marketing metrics."
        
    print(f"   Critique Score: {score}/10")
    print(f"   Editor Notes: {feedback}")
    
    return {
        "quality_score": score,
        "feedback": feedback
    }

# ==========================================
# 3. THE ROUTER (The Decision Gate)
# ==========================================
def gatekeeper_router(state: AgentState):
    print("🤖 [Router] Checking quality gate...")
    
    # If the score is 8 or higher, ship it! Otherwise, loop back to the writer.
    if state["quality_score"] >= 8:
        return "approved"
    else:
        return "needs_work"



In [3]:
# ==========================================
# 4. WIRING THE GRAPH
# ==========================================
builder = StateGraph(AgentState)

# Register our specialized agents
builder.add_node("writer", writer_agent)
builder.add_node("critic", critic_agent)

# Draw our execution paths
builder.add_edge(START, "writer")
builder.add_edge("writer", "critic")

# Add the conditional reflection loop
builder.add_conditional_edges(
    "critic",
    gatekeeper_router,
    {
        "needs_work": "writer",  # <--- THIS IS THE COOL PART: It loops backward!
        "approved": END
    }
)

# Compile the engine
orchestrator = builder.compile()
print("\n🎉 Reflection Engine Compiled Successfully!")


🎉 Reflection Engine Compiled Successfully!


In [5]:
# ==========================================
# 5. TEST RUN
# ==========================================
initial_state = {
    "topic": "Fast Databases",
    "current_draft": "",
    "feedback": "",
    "quality_score": 0,
    "revision_count": 0
}

final_output = orchestrator.invoke(initial_state)

print("\n==============================")
print("🚀 PROMOTED TO PRODUCTION:")
print(f"Final Copy: {final_output['current_draft']}")
print(f"Total Iterations Needed: {final_output['revision_count']}")
print("==============================")


✍️ [Writer] Current Revision: 1
   Draft 1: 'Our new software tool makes databases run super fast.'
🧐 [Critic] Analyzing the latest draft...
   Critique Score: 5/10
   Editor Notes: Too boring. Needs an emoji, a cooler product name, and the word 'latency'.
🤖 [Router] Checking quality gate...

✍️ [Writer] Current Revision: 2
   Received Feedback: "Too boring. Needs an emoji, a cooler product name, and the word 'latency'."
   Draft 2 (Revised): '⚡ QuantumDB: Rocket-fuel your SQL queries with zero latency.'
🧐 [Critic] Analyzing the latest draft...
   Critique Score: 9/10
   Editor Notes: Perfect. This hits all our enterprise marketing metrics.
🤖 [Router] Checking quality gate...

🚀 PROMOTED TO PRODUCTION:
Final Copy: ⚡ QuantumDB: Rocket-fuel your SQL queries with zero latency.
Total Iterations Needed: 2


In [6]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
# 1. Import the built-in in-memory checkpointer
from langgraph.checkpoint.memory import MemorySaver

# Define our state
class ChatState(TypedDict):
    user_name: str
    chat_history: Annotated[list, operator.add]

# Define a simple node
def greeting_node(state: ChatState):
    history = state["chat_history"]
    if not history:
        reply = f"Hello {state['user_name']}! Nice to meet you."
    else:
        reply = f"Welcome back, {state['user_name']}! You have sent {len(history) + 1} messages total."
    
    return {"chat_history": [reply]}

# Build the graph
builder = StateGraph(ChatState)
builder.add_node("greeter", greeting_node)
builder.add_edge(START, "greeter")
builder.add_edge("greeter", END)

# =========================================================
# THE TRANSFORMATION: Add Memory during Compilation
# =========================================================
memory_storage = MemorySaver()

# We pass the checkpointer straight into the compiler!
persistent_app = builder.compile(checkpointer=memory_storage)
print("💾 Persistent Graph Compiled with Memory Engine!")

💾 Persistent Graph Compiled with Memory Engine!


In [ ]:
# --- SESSION 1: Meeting Abhishek ---
config_1 = {"configurable": {"thread_id": "abhishek_session"}}

print("--- RUN 1 ---")
output1 = persistent_app.invoke({"user_name": "Abhishek", "chat_history": ["Message 1"]}, config=config_1)
print(f"Agent says: {output1['chat_history'][-1]}")

# --- SESSION 2: Running it again on the same thread ---
print("\n--- RUN 2 (Same Thread) ---")
# Notice we don't even need to pass the "user_name" anymore! It's saved in the checkpoint.
output2 = persistent_app.invoke({"chat_history": ["Message 2"]}, config=config_1)
print(f"Agent says: {output2['chat_history'][-1]}")

# --- SESSION 3: A completely different user thread ---
print("\n--- RUN 3 (Different Thread) ---")
config_2 = {"configurable": {"thread_id": "john_session"}}
output3 = persistent_app.invoke({"user_name": "John", "chat_history": ["Hello"]}, config=config_2)
print(f"Agent says: {output3['chat_history'][-1]}")